# 1. 인간 피드백 기반 강화학습 (RLHF)과 DPO (Direct Preference Optimization)

이전 차시까지 우리는 입력문장에 부합하는 대답을 생성하도록 정밀 학습을 시키는 파인튜닝(Full FT, LoRA) 기법을 실습했습니다.
하지만 AI가 유용하고(Helpful), 정직하며(Honest), 무해한(Harmless) 답변을 제공하도록 즉, **인간의 가치와 선호에 정렬(Alignment)** 되도록 하는 것은 단순한 분류나 텍스트 복제 방식의 학습으로는 달성하기 어렵습니다.

 LLM의 정렬을 위해 필수적으로 쓰이는 **RLHF** 와 최신 정렬 기법인 **DPO** 의 개념을 이해하고, 실습용 `trl` 라이브러리를 활용하여 GPT-2 모델을 직접 선호 정렬 학습시킵니다.

## 1. RLHF와 DPO의 핵심 이해

### 1) RLHF (Reinforcement Learning from Human Feedback) 란?
- 모델의 출력을 인간의 평가 기준에 부합하도록 지도하는 3단계 정렬 파이프라인입니다.
  1. **SFT (Supervised Fine-tuning)**: 지시어와 원하는 고품질 답변 데이터로 지도학습을 진행합니다.
  2. **RM (Reward Model, 보상 모델) 학습**: 동일한 질문에 대한 여러 답변을 인간이 선호도에 따라 순위를 매기고, 이를 바탕으로 질문-답변의 점수를 예측하는 보상 모델을 학습시킵니다.
  3. **PPO (Proximal Policy Optimization) 강화학습**: 2단계에서 만든 보상 모델의 점수를 극대화하도록 PPO 알고리즘을 사용해 LLM의 가중치를 갱신합니다.
- **한계**: 보상 모델 학습 및 PPO 강화학습을 결합하는 루프가 복잡하고, 학습이 매우 불안정하며 메모리 소모가 극도로 심합니다.

### 2) DPO (Direct Preference Optimization) 란?
- Stanford 대학 연구진이 제안한 기법으로, **별도의 보상 모델(Reward Model)과 복잡한 강화학습 PPO 과정 없이** , 사람이 선호한 답변(Chosen)과 거부한 답변(Rejected)만을 가지고 **수학적 손실 함수를 사용해 직접 정책 모델을 튜닝** 하는 방식입니다.
- 수학적 치환을 통해 RLHF의 복잡한 PPO 목적 함수를 단순한 이진 교차 엔트로피(Binary Cross-Entropy) 손실 함수로 매핑하여 학습 안전성과 효율성을 대폭 끌어올렸습니다.

In [1]:
from datasets import Dataset
from trl import DPOTrainer,DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


## 2. DPO용 가상 선호 데이터셋 정의
DPO 학습 데이터셋은 반드시 아래 세 가지 열을 갖춰야 합니다:
1. `prompt`: 모델에 전달하는 질문/지시문
2. `chosen`: 사람이 선호하는 긍정 답변 (Target Winner)
3. `rejected`: 사람이 선호하지 않는 잘못되거나 부정적인 답변 (Target Loser)

In [2]:
dpo_train_data = [
    {
        "prompt": "Review: This movie is a masterpiece. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: I hated this film. Total waste of time. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    },
    {
        "prompt": "Review: Outstanding acting and brilliant plot. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Terrible acting and boring screenplay. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"

    }
]

dpo_test_data = [
    {
        "prompt": "Review: The movie was mediocre, but the ending was spectacular! Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    }
]

train_dataset = Dataset.from_list(dpo_train_data)
test_dataset = Dataset.from_list(dpo_test_data)
print("DPO dataset initialized!")

DPO dataset initialized!


## 3. 베이스 모델 로드 및 LoRA 설정 적용
- DPO 학습 대상 모델로 디코더 형태의 모델인 `gpt2`를 로드합니다.
- 학습 파라미터 소모를 대폭 줄이기 위해 LoRA 기법을 결합하여 어댑터만 튜닝하도록 구성합니다.

### AutoModelForSeq2SeqLM, AutoModelForCausalLM 비교
- Encoder-Decoder               Decoder Only
-   T5, BART                    GPT,LIma
- 입력 -> Encoder                입력->Decoder 
- 벡터 -> Decoder- 출력           -> 다른 토큰으로 예측 -> 다른 토큰으로 예측

In [3]:
model = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]